# 02 — Fama-MacBeth Cross-Sectional Regressions

Replicates Table 3 of Carr & Wu (2009) and extends to post-2015 data.

**Specification:**
```
r_{i,t+1} = λ₀_t + λ₁_t · VRP_{i,t} + λ₂_t · β_{i,t} + λ₃_t · log(ME)_{i,t} + ε
```
t-statistics use Newey-West HAC errors (6 lags).

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from src.fama_macbeth import fama_macbeth, rolling_fama_macbeth

DATA_DIR = '../data'

vrp = pd.read_parquet(f'{DATA_DIR}/vrp_panel.parquet')
returns = pd.read_parquet(f'{DATA_DIR}/returns_panel.parquet')
market_cap = pd.read_parquet(f'{DATA_DIR}/market_cap_panel.parquet')

log_me = np.log(market_cap)

# Rolling 12-month beta (vs equal-weighted market)
mkt = returns.mean(axis=1)
beta = returns.rolling(12).apply(
    lambda col: np.cov(col, mkt.loc[col.index])[0, 1] / mkt.loc[col.index].var(),
    raw=False
)

print('Data loaded.')

ModuleNotFoundError: No module named 'matplotlib'

## Full-sample Fama-MacBeth

In [3]:
factors = {'VRP': vrp, 'beta': beta, 'log_me': log_me}

full_sample = fama_macbeth(returns, factors, nw_lags=6)
print('=== Full sample ===')
print(full_sample.round(4))

NameError: name 'vrp' is not defined

## Sub-period regressions

In [4]:
sub_periods = {
    '1996-2009 (in-sample)': ('1996-01-01', '2009-12-31'),
    '2010-2015 (OOS)':       ('2010-01-01', '2015-12-31'),
    '2016-2022 (post-pub)':  ('2016-01-01', '2022-12-31'),
    '2023-2025 (extended)':  ('2023-01-01', '2025-12-31'),
}

results = {}
for label, (start, end) in sub_periods.items():
    mask = (returns.index >= start) & (returns.index <= end)
    sub_ret = returns.loc[mask]
    sub_fac = {k: v.loc[mask] for k, v in factors.items()}
    try:
        res = fama_macbeth(sub_ret, sub_fac, nw_lags=6)
        results[label] = res.loc['VRP', ['lambda', 't_stat', 'n_periods']]
    except ValueError as e:
        print(f'{label}: insufficient data ({e})')

summary = pd.DataFrame(results).T
summary.columns = ['λ₁ (VRP)', 't-stat', 'N months']
print('\n=== Sub-period VRP coefficient ===')
print(summary.round(4))

NameError: name 'returns' is not defined

## Rolling λ₁ — visualising the decay

In [5]:
rolling = rolling_fama_macbeth(returns, factors, window=36, nw_lags=6)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(rolling.index, rolling['VRP_lambda'], color='#185FA5', linewidth=1.5)
axes[0].axhline(0, color='#888780', linewidth=0.7, linestyle='--')
axes[0].set_ylabel('λ₁ (VRP slope)', fontsize=11)
axes[0].set_title('Rolling 36-month Fama-MacBeth VRP coefficient', fontsize=12)

axes[1].plot(rolling.index, rolling['VRP_tstat'], color='#BA7517', linewidth=1.5)
axes[1].axhline(1.96, color='#639922', linewidth=0.7, linestyle='--', label='5% threshold')
axes[1].axhline(0, color='#888780', linewidth=0.7, linestyle='--')
axes[1].set_ylabel('t-statistic', fontsize=11)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('../data/rolling_lambda.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'rolling_fama_macbeth' is not defined